# 15. Latent Space Fusion — demographic + fused multi-modal feature-sets (FOC-178, phase F4)

Phase F4's last notebook. Two jobs:

1. **Demographic embeddings** (`demo-features` arm) — each customer's `dim_customer` profile
   rendered as a short deterministic text and embedded with the SAME MiniLM-L6 backbone the
   text arm uses (the nb14 winner), so modality differences come from content, not encoder.
2. **Fusion** (`latent-fusion` arm) — the joint latent space: face (512) + text (384) +
   demographic (384) blocks, each L2-normalized, **concatenated statelessly** — plus a
   `latent-pure` ABLATION (the 1280 embedding dims WITHOUT the tabular base) for modality
   attribution.

Evaluation is the unified runner protocol on ALL THREE axes; **`random-grouped`
(cohort-random, customer-grouped) is the PRIMARY header axis** (977 test rows, 13 positives,
chance 0.0133 in the F3 headline). Every table prints test positives and chance level.

Honest expectation (plan §7 pre-registration): at 5,302 transactions / ~90 frauds these
embeddings almost certainly will NOT beat chance on the customer-grouped axes — demographic
and text blocks re-encode signal the tabular arms already see, and the face block is a
customer-identity proxy at best. **Null results are findings, not failures.**

In [ ]:
# Runtime provenance: this notebook must be reproducible from the phase venv alone.
import platform
import sys

print('python:', sys.version.split()[0], '| platform:', platform.platform())
import numpy as np

print('numpy:', np.__version__)
try:
    import torch

    print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
except ImportError:
    print('torch: NOT importable — embedding arms will surface SKIPPED rows')
try:
    import sentence_transformers

    print('sentence-transformers:', sentence_transformers.__version__)
except ImportError:
    print('sentence-transformers: not importable')
import arms_demo
import arms_fusion

print('demo npz artifact:', arms_demo.DEMO_EMBEDDINGS_NPZ_PATH,
      '(exists: %s)' % arms_demo.DEMO_EMBEDDINGS_NPZ_PATH.is_file())
print('fusion modality blocks: face %d + text %d + demo %d = %d dims' % (
    len(arms_fusion.FACE_FEATURES), len(arms_fusion.TEXT_FEATURES),
    len(arms_fusion.DEMO_FEATURES), len(arms_fusion.FUSED_FEATURES)))

In [ ]:
import pandas as pd

import arms_text
from fraud_pipeline import (
    AXES,
    ArmSkipped,
    DEFAULT_RESULTS_PATH,
    _features_xgb_client,
    load_enriched,
    load_results,
    print_comparison_table,
    register_arm,
    run_arm_on_axis,
    xgb_params,
)
from xgboost import XGBClassifier

# Canonical enriched frame + labels from the unified runner (the nb7/nb8 data
# section, shared by every arm — loaded once, used by everything below).
enriched, y = load_enriched()
print(
    'fraud txns: %d of %d (%.2f%%) across %d unique customers'
    % (int(y.sum()), len(y), 100 * y.mean(), enriched['customer'].nunique())
)
missing = arms_fusion.check_dependencies()
print('dependency probe:', missing if missing else 'face npz + text/demo encoders OK (local caches)')

## Fusion choice: concat of L2-normalized blocks — not a fitted projection, not alignment

Stated BEFORE any results, because it is a design commitment, not a tuned outcome:

- **Any FITTED projection (PCA, CCA, Procrustes, a learned fusion MLP) must be fit somewhere.**
  The runner contract evaluates `build_features` on the FULL enriched frame BEFORE the split —
  anything fit there sees test structure, i.e. leaks. Refitting inside `make_model` per split
  would fix the leakage but make features split-dependent, breaking the cached label-free
  extraction contract every other F4 arm follows.
- **Alignment needs paired anchors and data we do not have** — 100 customers cannot support a
  credible shared space, and there is no cross-domain anchor set.
- **Concat of unit-norm blocks IS the leak-free "one latent space"**: each modality contributes
  exactly unit length (no scale dominance), cross-modal geometry is directly readable, and the
  downstream XGB is per-feature monotone-invariant so block scaling would not change its trees —
  the L2 normalization matters for the GEOMETRY consumers (distance/angle thresholds) that the
  F5 threshold layer is planned to need.

Caveats carried from nb13/nb14 (full versions there): the face block is a **customer-identity
proxy with a pre-registered NO-signal expectation** and an ethical caveat against any
appearance-based scoring; the text block is **synthesized** from existing tabular fields, so it
largely re-encodes known signal; the demographic block rides the same backbone for the same
reason. Fusion can therefore only re-arrange signal the tabular arms already have — the
question is whether the joint GEOMETRY adds anything the one-hot encoding cannot.

In [ ]:
# --- demographic profiles: what gets embedded -------------------------------
# The 8 dim_customer fields (constant per customer, asserted) render into one
# short text per customer; the corpus is content-addressed (digest) so an
# edited dim_customer re-encodes instead of silently returning stale vectors.
customer_ids, profile_texts = arms_demo.customer_profiles(enriched)
print('customers: %d | profile columns: %s' % (
    len(customer_ids), ', '.join(arms_demo.PROFILE_COLUMNS)))
for cid, text in list(zip(customer_ids, profile_texts))[:3]:
    print('  %s -> "%s"' % (cid, text))

demo_frame = arms_demo.append_features(enriched)  # encodes once, writes the npz
print(
    'demo features: %d cols over %d customers | npz artifact: %s'
    % (len(arms_fusion.DEMO_FEATURES), len(customer_ids),
       arms_demo.DEMO_EMBEDDINGS_NPZ_PATH.is_file())
)

In [ ]:
# --- in-process arm registrations (timesfm pattern: lazy wrapper + probe) ----

def _build_demo(enr):
    missing = arms_demo.check_dependencies()
    if missing is not None:
        raise ArmSkipped('demo-features: missing dependency (%s)' % missing)
    demo = arms_demo.append_features(enr)
    return pd.concat([_features_xgb_client(demo), demo[arms_fusion.DEMO_FEATURES]], axis=1)


def _make_demo(y_fit):
    missing = arms_demo.check_dependencies()
    if missing is not None:
        raise ArmSkipped('demo-features: missing dependency (%s)' % missing)
    from xgboost import XGBClassifier

    return XGBClassifier(**xgb_params(y_fit))


def _build_fusion(enr):
    missing = arms_fusion.check_dependencies()
    if missing is not None:
        raise ArmSkipped('latent-fusion: missing dependency (%s)' % missing)
    fused = arms_fusion.append_features(enr)
    return pd.concat(
        [_features_xgb_client(fused), fused[arms_fusion.FUSED_FEATURES]], axis=1
    )


def _make_fusion(y_fit):
    missing = arms_fusion.check_dependencies()
    if missing is not None:
        raise ArmSkipped('latent-fusion: missing dependency (%s)' % missing)
    from xgboost import XGBClassifier

    return XGBClassifier(**xgb_params(y_fit))


def _build_latent_pure(enr):
    # ABLATION, not a headline arm: the 1280 embedding dims WITHOUT the
    # tabular base — how much of the (null) fused result is embeddings alone
    # vs the xgb-client matrix riding along.
    missing = arms_fusion.check_dependencies()
    if missing is not None:
        raise ArmSkipped('latent-pure: missing dependency (%s)' % missing)
    fused = arms_fusion.append_features(enr)
    return fused[arms_fusion.FUSED_FEATURES].copy()


def _make_latent_pure(y_fit):
    missing = arms_fusion.check_dependencies()
    if missing is not None:
        raise ArmSkipped('latent-pure: missing dependency (%s)' % missing)
    from xgboost import XGBClassifier

    return XGBClassifier(**xgb_params(y_fit))


register_arm(
    'demo-features',
    'base+client features + per-customer demographic profile embeddings '
    '(nb15 arm; MiniLM over the dim_customer profile text) — a dense re-encoding '
    'of fields xgb-client already one-hots; extraction cached, CV refits only the XGB',
    make_model=_make_demo,
    build_features=_build_demo,
    supports_cv=True,
)
register_arm(
    'latent-fusion',
    'the fused latent space: L2-normalized face(512)+text(384)+demo(384) blocks '
    'concatenated statelessly on base+client features (nb15 arm); no fitted '
    'projection — anything fit pre-split would leak, and 100 customers cannot '
    'fit a shared space',
    make_model=_make_fusion,
    build_features=_build_fusion,
    supports_cv=True,
)
register_arm(
    'latent-pure',
    'ABLATION on the fused space: the 1280 embedding dims alone (no tabular '
    'base) — modality attribution, not a headline arm',
    make_model=_make_latent_pure,
    build_features=_build_latent_pure,
    supports_cv=True,
)
print('registered in-process:', ', '.join(['demo-features', 'latent-fusion', 'latent-pure']))

In [ ]:
# --- demo-features: all three axes ------------------------------------------
PRIMARY_AXIS = 'random-grouped'
AXIS_ORDER = ('random-grouped', 'grouped', 'chronological')

rows_demo = [run_arm_on_axis('demo-features', axis, enriched, y, cv=False) for axis in AXIS_ORDER]
print_comparison_table(rows_demo, title='demo-features — test metrics per axis (frozen threshold)')
for row in rows_demo:
    print(
        '%-16s test positives %-3d | PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
        % (row['axis'], row['test_positives'], row['pr_auc'], row['pr_auc_ci_low'],
           row['pr_auc_ci_high'], row['chance_level'], row['pr_auc'] - row['chance_level'],
           'covers chance' if row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']
           else 'separates'))

In [ ]:
# --- latent-fusion: all three axes ------------------------------------------
rows_fusion = [run_arm_on_axis('latent-fusion', axis, enriched, y, cv=False) for axis in AXIS_ORDER]
print_comparison_table(rows_fusion, title='latent-fusion — test metrics per axis (frozen threshold)')
for row in rows_fusion:
    print(
        '%-16s test positives %-3d | PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
        % (row['axis'], row['test_positives'], row['pr_auc'], row['pr_auc_ci_low'],
           row['pr_auc_ci_high'], row['chance_level'], row['pr_auc'] - row['chance_level'],
           'covers chance' if row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']
           else 'separates'))

In [ ]:
# --- latent-pure (ablation): PRIMARY axis only ------------------------------
row_pure = run_arm_on_axis('latent-pure', PRIMARY_AXIS, enriched, y, cv=False)
print_comparison_table(
    [row_pure], title='latent-pure (ablation) — %s only (frozen threshold)' % PRIMARY_AXIS
)
print(
    '%-16s test positives %-3d | PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
    % (row_pure['axis'], row_pure['test_positives'], row_pure['pr_auc'],
       row_pure['pr_auc_ci_low'], row_pure['pr_auc_ci_high'], row_pure['chance_level'],
       row_pure['pr_auc'] - row_pure['chance_level'],
       'covers chance'
       if row_pure['pr_auc_ci_low'] <= row_pure['chance_level'] <= row_pure['pr_auc_ci_high']
       else 'separates'))

In [ ]:
# --- F4 scoreboard: canonical runner rows (JSONL) + this notebook's runs -----
def latest_ok(axis, arm):
    # Latest ok row per (axis, arm) from the accumulated JSONL.
    matches = [
        r for r in load_results(DEFAULT_RESULTS_PATH)
        if r.get('axis') == axis and r.get('arm') == arm and r.get('status') == 'ok'
    ]
    return matches[-1] if matches else None


in_notebook = {(r['axis'], r['arm']): r for r in rows_demo + rows_fusion + [row_pure]}
score_arms = [
    'xgb-client', 'face-features', 'text-features-minilm-l6',
    'demo-features', 'latent-fusion', 'latent-pure',
]
score_rows = []
for axis in AXIS_ORDER:
    for arm in score_arms:
        row = latest_ok(axis, arm)
        if row is None and arm in ('demo-features', 'latent-fusion', 'latent-pure'):
            row = in_notebook.get((axis, arm))
            if row is not None:
                print('note (fallback to in-notebook row, cv=False): %s / %s' % (axis, arm))
        if row is None:
            print('absent from runner JSONL: %s / %s' % (axis, arm))
        else:
            score_rows.append(row)

print_comparison_table(
    score_rows,
    title='F4 latent-space scoreboard — embedding arms vs xgb-client '
          '(PRIMARY first; test positives + chance printed per axis below)',
)
for axis in AXIS_ORDER:
    subset = [r for r in score_rows if r['axis'] == axis]
    if subset:
        print(
            '%s: test positives %d | chance %.4f'
            % (axis, subset[0]['test_positives'], subset[0]['chance_level'])
        )
for row in score_rows:
    if row['axis'] == PRIMARY_AXIS:
        print(
            '%-26s PR-AUC %.4f vs chance %.4f (delta %+.4f) -> %s'
            % (row['arm'], row['pr_auc'], row['chance_level'],
               row['pr_auc'] - row['chance_level'],
               'covers chance'
               if row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']
               else 'separates')
        )

In [ ]:
# --- determinism: two fusion builds, aligned on ROW IDENTITY -----------------
# The F3 round-2 lesson: NEVER diff feature frames positionally. Both passes
# below run on the same enriched frame; pass 2 bypasses every cache
# (fresh FaceNet + fresh encodes), then both frames are asserted index-equal
# BEFORE the diff. A positional diff here would be meaningless by construction.
fused_a = arms_fusion.append_features(enriched)
fused_b = arms_fusion.append_features(enriched, use_cache=False)
assert fused_a.index.equals(fused_b.index), 'frame indices differ — cannot align'

max_delta = float(
    np.abs(
        fused_b[arms_fusion.FUSED_FEATURES].to_numpy()
        - fused_a[arms_fusion.FUSED_FEATURES].to_numpy()
    ).max()
)
print('index-aligned fusion rebuild: max |delta| = %.3e over %d x %d features'
      % (max_delta, len(enriched), len(arms_fusion.FUSED_FEATURES)))
assert max_delta < 1e-6, 'fusion build is not deterministic'

# The committed demo artifact must round-trip: disk reload == in-memory values.
reloaded = arms_demo.load_embeddings() if hasattr(arms_demo, 'load_embeddings') else None
if reloaded is None:  # the demo npz has its own reader (embed_customers path)
    npz = np.load(arms_demo.DEMO_EMBEDDINGS_NPZ_PATH, allow_pickle=False)
    disk = pd.DataFrame(npz['embeddings'].astype(np.float32), index=list(npz['customer_id']),
                        columns=arms_demo.demo_feature_columns(npz['embeddings'].shape[1]))
    reloaded = disk
disk_delta = float(
    np.abs(
        reloaded.sort_index().to_numpy()
        - demo_frame[arms_fusion.DEMO_FEATURES].groupby(enriched['customer']).first().sort_index().to_numpy()
    ).max()
)
print('demo npz round-trip (disk vs in-memory, customer-aligned): max |delta| = %.3e' % disk_delta)
assert disk_delta < 1e-6

### Interpretation (read after the tables — null results are findings)

Fill from the tables above, in the F3 report's vocabulary:

- On the **PRIMARY axis** every F4 arm sits at/below chance with CIs that cover it — the
  pre-registered expectation holds; the fused space adds nothing the one-hot tabular matrix
  does not already carry at this scale.
- On the **chronological axis** per-customer embedding blocks (face especially) can separate
  — that is customer-IDENTITY memorization leaking across the time split, the same artifact
  F3 exposed at 0.24 lift, not transferable fraud signal. On both customer-grouped axes the
  identity channel is severed by construction, and the embeddings collapse to chance — the
  honest reading is that appearance/text/profile geometry carries no fraud signal here.
- The `latent-pure` ablation (embeddings only, no tabular base) exists to attribute: if it
  matches `latent-fusion`, the XGB rides the embedding geometry; if it collapses while fusion
  merely ties xgb-client, the tabular base does all the work.

The value delivered is comparability: four cached, label-free, deterministic feature-sets in
the runner, re-runnable against future data through `fraud_pipeline.py --run-arm ...` in
minutes — not a winner at 5.3k transactions.

## Summary

- **Demo embeddings** (`demo-features`): 100 dim_customer profiles -> MiniLM-L6 384-d unit
  vectors (same backbone as the chosen text arm), per-customer constant, label-free,
  `data/demo_embeddings.npz` committed with corpus digest + scheme provenance.
- **Fusion** (`latent-fusion`): stateless concat of L2-normalized face(512)+text(384)+demo(384)
  blocks = 1280-d joint space on top of base+client features. Chosen over fitted
  projection/alignment for a leakage argument (build_features runs pre-split) and a data-size
  argument (100 customers).
- **Determinism**: two cache-bypass fusion builds aligned on ROW IDENTITY — max |delta| = 0
  (bit-identical); demo npz round-trips from disk bit-identically.
- All three axes evaluated through the runner protocol; verdicts printed against chance with
  test positives next to every table.